# ReAct agent with Weather tool
Agent that provides weather information

### SETUP

In [ ]:
%pip install -U langchain langchain-core langchain-community langchain-ollama --quiet

<div style="background-color: #e6f2ff; padding: 15px; border-left: 6px solid #0066cc; color: #004499;">
<h2>Notes on new agent architecture 💡</h2>
The <code>create_agent</code> is the newer LangGraph agent (as opposed to the old "agent_executor"):

- This uses a "state graph" for the agent to keep track of tool calls.
- They are recorded in "messages" ("tool_call" in "AIMessage" and "ToolMessage").
- There is no need to create a "scratchpad". Scratchpads are needed for legacy agent constructors like "AgentExecutor", "create_tool_calling_agent", "create_react_agent".

In this new architecture we distinguish between two specific types of memory:
- Short-Term Memory (Thread-level): <code>InMemorySaver</code> remembers everything within a specific thread_id.
- Long-Term Memory (Cross-thread): <code>InMemoryStore</code> remebers across threads. Ex: If I want the agent to remember that "User A likes metric units" even when starting a brand new chat (thread_id="002").
</div>

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

Read credentials

In [36]:
import os
from dotenv import load_dotenv
load_dotenv()
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
#TAVILY_API_KEY = tvly-dev-xxxxxxx

Instantiate the model

In [37]:
llm = ChatOllama(
    model="gemma4:e4b",
    temperature=0.1,
)

Tavily section

In [ ]:
%pip install tavily-python

In [40]:
from tavily import TavilyClient
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

Function that uses Tavily to search for current weather at a specific city. This will be a tool for the model

In [41]:
@tool
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    response = tavily_client.search(query=f"current weather in {city}", search_depth="basic")
    return response['results']

Give the agent "memory". To make it persistent through app restarts I can use `PostgresSaver()` or `SqliteSaver()`

In [42]:
checkpointer = InMemorySaver()

Create the agent

In [43]:
SYSTEM_PROMPT = """You are an expert weather forecaster.
You have access to a tool:
- get_weather: use this to get the weather for a specific location"""

agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer
)

### TESTING
Let's try it out

In [44]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "My name is Tom. I live in Brisbane"}]},
    config={"configurable": {"thread_id": "brisbane-weather"}}
)

In [45]:
import pprint
pprint.pprint(response)

{'messages': [HumanMessage(content='My name is Tom. I live in Brisbane', additional_kwargs={}, response_metadata={}, id='08ad8e11-4c1f-4cdd-836c-3c79a3be480c'),
              AIMessage(content="Hello Tom! It's nice to meet you. I can help you with the weather in Brisbane if you'd like to know what it's like there!", additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-05-10T07:11:58.7035624Z', 'done': True, 'done_reason': 'stop', 'total_duration': 15551842700, 'load_duration': 10277694000, 'prompt_eval_count': 100, 'prompt_eval_duration': 150447700, 'eval_count': 405, 'eval_duration': 4661400000, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019e10ba-442d-7f61-a38a-8bbe9134b192-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 100, 'output_tokens': 405, 'total_tokens': 505})]}


In [46]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in my city? Please address me by my name if you remember it."}]},
    config={"configurable": {"thread_id": "brisbane-weather"}}
)

In [47]:
import pprint
pprint.pprint(response)

{'messages': [HumanMessage(content='My name is Tom. I live in Brisbane', additional_kwargs={}, response_metadata={}, id='08ad8e11-4c1f-4cdd-836c-3c79a3be480c'),
              AIMessage(content="Hello Tom! It's nice to meet you. I can help you with the weather in Brisbane if you'd like to know what it's like there!", additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-05-10T07:11:58.7035624Z', 'done': True, 'done_reason': 'stop', 'total_duration': 15551842700, 'load_duration': 10277694000, 'prompt_eval_count': 100, 'prompt_eval_duration': 150447700, 'eval_count': 405, 'eval_duration': 4661400000, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019e10ba-442d-7f61-a38a-8bbe9134b192-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 100, 'output_tokens': 405, 'total_tokens': 505}),
              HumanMessage(content='What is the weather in my city? Please address me by my name if you remember it.'

In [48]:
print(response["messages"][-1].content)

Hello Tom! The weather in Brisbane right now is **Overcast**.

Here are the details for you:

*   **Temperature:** 22.0°C (71.6°F)
*   **Feels Like:** 24.6°C (76.2°F)
*   **Wind:** Blowing from the ESE at 12.6 kph (7.8 mph)
*   **Humidity:** 65%
*   **Chance of Rain:** There is a 62% chance of rain.

It looks like a bit cloudy with a chance of some drizzle! Let me know if you need a forecast for another day or location.


In [49]:
from IPython.display import Markdown
Markdown(response["messages"][-1].content)

Hello Tom! The weather in Brisbane right now is **Overcast**.

Here are the details for you:

*   **Temperature:** 22.0°C (71.6°F)
*   **Feels Like:** 24.6°C (76.2°F)
*   **Wind:** Blowing from the ESE at 12.6 kph (7.8 mph)
*   **Humidity:** 65%
*   **Chance of Rain:** There is a 62% chance of rain.

It looks like a bit cloudy with a chance of some drizzle! Let me know if you need a forecast for another day or location.

In [50]:
response["messages"][-1].usage_metadata

{'input_tokens': 1208, 'output_tokens': 398, 'total_tokens': 1606}

In [51]:
for tool_call in response["messages"][-3].tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")
    print(f"ID: {tool_call['id']}")

Tool: get_weather
Args: {'city': 'Brisbane'}
ID: d2494107-3f3b-4d1e-af88-4d6a7140eed9


In [52]:
for m in response["messages"]:
    print(m.pretty_print())

================================ Human Message =================================

My name is Tom. I live in Brisbane
None
================================== Ai Message ==================================

Hello Tom! It's nice to meet you. I can help you with the weather in Brisbane if you'd like to know what it's like there!
None
================================ Human Message =================================

What is the weather in my city? Please address me by my name if you remember it.
None
================================== Ai Message ==================================
Tool Calls:
  get_weather (d2494107-3f3b-4d1e-af88-4d6a7140eed9)
 Call ID: d2494107-3f3b-4d1e-af88-4d6a7140eed9
  Args:
    city: Brisbane
None
================================= Tool Message =================================
Name: get_weather

[{"title": "Weather in Brisbane", "url": "https://www.weatherapi.com/", "content": "{'location': {'name': 'Brisbane', 'region': 'Queensland', 'country': 'Australia', 'lat': -27